# 01. Data Cleaning & Feature Engineering

**Th?nh vi?n ph? tr?ch:** Th?nh vi?n 1 - Data Engineering & Preprocessing

**M?c ti?u quy tr?nh x? l? d? li?u th?c t? (`1.csv`):**
1. **Team Win%:** T?ch b?ng t? l? th?ng c?a t?ng ??i b?ng theo m?a gi?i (`team_win_pct`).
2. **L?c k? nguy?n hi?n ??i:** Gi? c?c m?a gi?i $\ge 1980$ v? lo?i tr? m?a gi?i 1999 (b? h?y All-Star do Lockout).
3. **T?o nh?n All-Star & X? l? Trade gi?a m?a:** K?t h?p nh?n `replaced` v? `IS_ALLSTAR`; x?c ??nh ??i b?ng ch?nh (`main_tm`) v? ?u ti?n gi? d?ng t?ng h?p (`TOT`, `2TM`, `3TM`, `4TM`, `5TM`) ?? m?i c?p `(player_id, season)` l? duy nh?t.
4. **Gh?p t? l? th?ng:** Gh?p `team_win_pct` t??ng ?ng v?i ??i b?ng ch?nh c?a c?u th?.
5. **T?nh s? l?n All-Star qu? kh? (`prev_allstar_count`):** Th?ng k? s? l?n ???c ch?n All-Star tr??c m?a gi?i hi?n t?i (tr?nh data leakage).
6. **L?c nhi?u & X? l? Missing Value:** L?c c?c c?u th? c? $G \ge 15$ v? $MP/G \ge 10.0$; ?i?n gi? tr? 0.0 cho c?c t? l? n?m thi?u.
7. **Era Adjustment (Chu?n h?a Z-Score theo m?a):** T?nh Z-score n?i b? t?ng m?a gi?i cho 6 ch? s?: `pts_per_game`, `ast_per_game`, `trb_per_game`, `per`, `ws`, `vorp`.
8. **Chia t?p d? li?u:** L?u tr? `data/processed/nba_clean.csv` v? chia th?nh `train.csv` (<=2016), `val.csv` (2017-2019), `test.csv` (>=2020).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.preprocessing import clean_raw_nba_data, split_by_season, build_preprocessing_pipeline

print("Imports th?nh c?ng t? src.preprocessing!")

## B??c 1: N?p file th? `data/raw/1.csv`

In [ ]:
raw_csv_path = ROOT_DIR / "data" / "raw" / "1.csv"
print(f"??c d? li?u t?: {raw_csv_path}")

df_raw = pd.read_csv(raw_csv_path, low_memory=False)
print(f"K?ch th??c d? li?u th?: {df_raw.shape}")
df_raw.head(3)

## B??c 2: T?ch b?ng t? l? th?ng c?a ??i b?ng (Team Win%)
Tr?ch xu?t t? l? th?ng `w / (w + l)` c?a t?ng ??i b?ng trong t?ng m?a gi?i t? n?m 1980.

In [ ]:
team_rows = df_raw[
    df_raw['player_id'].isnull() & 
    df_raw['abbreviation'].notnull() & 
    df_raw['w'].notnull() & 
    (df_raw['lg'] == 'NBA') & 
    (df_raw['season'] >= 1980)
].copy()

team_rows['team_win_pct'] = team_rows['w'] / (team_rows['w'] + team_rows['l'])
team_win_lookup = team_rows[['season', 'abbreviation', 'team_win_pct']].drop_duplicates(subset=['season', 'abbreviation'])
print(f"S? l??ng b?n ghi t? l? th?ng ??i b?ng: {len(team_win_lookup)}")
team_win_lookup.head()

## B??c 3: L?c c?u th? NBA t? m?a 1980 (lo?i tr? m?a 1999 Lockout)

In [ ]:
players = df_raw[
    df_raw['player_id'].notnull() & 
    df_raw['pts_per_game'].notnull() & 
    (df_raw['lg'] == 'NBA') & 
    (df_raw['season'] >= 1980) &
    (df_raw['season'] != 1999)
].copy()

print(f"S? b?n ghi c?u th? sau l?c: {len(players)}")

## B??c 4: T?o nh?n All_Star & X? l? c?u th? ??i ??i gi?a m?a (Deduplicate Traded Players)
- G?p `replaced` v? `IS_ALLSTAR` th?nh nh?n `All_Star`.
- T?m ??i b?ng thi ??u nhi?u tr?n nh?t l?m `main_tm`.
- Gi? d?ng t?ng (`TOT`, `2TM`, `3TM`, `4TM`, `5TM`) ?? m?i c?p `(player_id, season)` l? duy nh?t.

In [ ]:
players['All_Star'] = np.where((players['replaced'].notnull()) | (players['IS_ALLSTAR'] == 1.0), 1, 0)
players['All_Star'] = players.groupby(['player_id', 'season'])['All_Star'].transform('max')

multi_tm_codes = ['TOT', '2TM', '3TM', '4TM', '5TM']
real_team_rows = players[~players['team'].isin(multi_tm_codes)].copy()
main_team = (
    real_team_rows.sort_values('g', ascending=False)
    .drop_duplicates(subset=['player_id', 'season'], keep='first')[['player_id', 'season', 'team']]
    .rename(columns={'team': 'main_tm'})
)

players['is_total_row'] = np.where(players['team'].isin(multi_tm_codes), 0, 1)
players = (
    players.sort_values(['player_id', 'season', 'is_total_row', 'g'], ascending=[True, True, True, False])
    .drop_duplicates(subset=['player_id', 'season'], keep='first')
    .copy()
)

print(f"S? d?ng c?u th? sau khi kh? tr?ng l?p trade: {len(players)}")

## B??c 5: Gh?p t? l? th?ng ??i & T?nh s? l?n All-Star qu? kh? (`prev_allstar_count`)

In [ ]:
# Gh?p team_win_pct qua main_tm
players = pd.merge(players, main_team, on=['player_id', 'season'], how='left')
players['main_tm'] = players['main_tm'].fillna(players['team'])
players = pd.merge(players, team_win_lookup, left_on=['season', 'main_tm'], right_on=['season', 'abbreviation'], how='left')

# T?nh s? l?n All-Star t?ch l?y tr??c m?a gi?i hi?n t?i
players['season'] = players['season'].astype(int)
players = players.sort_values(['player_id', 'season']).reset_index(drop=True)
players['prev_allstar_count'] = (
    players.groupby('player_id')['All_Star']
    .transform(lambda x: x.cumsum().shift(1).fillna(0))
    .astype(int)
)
print("Ph?n b? prev_allstar_count:")
print(players['prev_allstar_count'].value_counts().head())

## B??c 6: L?c nhi?u ($G \ge 15, MP/G \ge 10.0$) & X? l? Missing Values

In [ ]:
players_clean = players[(players['g'] >= 15) & (players['mp_per_game'] >= 10.0)].copy()

for pct_col in ['fg_percent', 'x3p_percent', 'ft_percent', 'ts_percent']:
    players_clean[pct_col] = players_clean[pct_col].fillna(0.0)

players_clean['gs'] = players_clean['gs'].fillna(0).astype(int)
players_clean['g'] = players_clean['g'].astype(int)
players_clean['age'] = players_clean['age'].astype(int)

print(f"S? l??ng c?u th? sau khi l?c nhi?u: {len(players_clean)}")

## B??c 7: Chu?n h?a Z-Score theo t?ng m?a gi?i (Era Adjustment)
Kh? s? kh?c bi?t v? nh?p ?? thi ??u (pace) v? trung b?nh ?i?m s? gi?a c?c th?p ni?n.

In [ ]:
zscore_features = ['pts_per_game', 'ast_per_game', 'trb_per_game', 'per', 'ws', 'vorp']
for col in zscore_features:
    mean = players_clean.groupby('season')[col].transform('mean')
    std = players_clean.groupby('season')[col].transform('std').replace(0, 1)
    players_clean[f'{col}_zscore'] = ((players_clean[col] - mean) / std).fillna(0.0)

final_cols = [
    'player_id', 'player', 'season', 'pos', 'age', 'team', 'main_tm',
    'g', 'gs', 'mp_per_game', 'pts_per_game', 'ast_per_game', 'trb_per_game',
    'stl_per_game', 'blk_per_game', 'tov_per_game',
    'fg_percent', 'x3p_percent', 'ft_percent',
    'per', 'ts_percent', 'usg_percent', 'ws', 'bpm', 'vorp',
    'team_win_pct', 'prev_allstar_count',
    'pts_per_game_zscore', 'ast_per_game_zscore', 'trb_per_game_zscore',
    'per_zscore', 'ws_zscore', 'vorp_zscore',
    'All_Star'
]

nba_clean = players_clean[final_cols].copy()
print(f"B?ng d? li?u s?ch ho?n ch?nh: {nba_clean.shape}")
print(f"S? l??ng All-Star th?c t?: {nba_clean['All_Star'].sum()} / {len(nba_clean)} ({nba_clean['All_Star'].mean():.2%})")

## B??c 8: L?u tr? `data/processed/nba_clean.csv` v? Ph?n chia Train/Val/Test

In [ ]:
processed_dir = ROOT_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# L?u file nba_clean.csv
clean_csv_path = processed_dir / "nba_clean.csv"
nba_clean.to_csv(clean_csv_path, index=False)
print(f"?? l?u: {clean_csv_path}")

# Ph?n chia th?i gian: Train <= 2016, Val 2017-2019, Test >= 2020
train_df, val_df, test_df = split_by_season(nba_clean, train_end=2016, val_end=2019, season_col="season")

train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "val.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print(f"Train: {train_df.shape} (M?a {train_df['season'].min()}-{train_df['season'].max()}), All-Stars: {train_df['All_Star'].sum()}")
print(f"Val:   {val_df.shape} (M?a {val_df['season'].min()}-{val_df['season'].max()}), All-Stars: {val_df['All_Star'].sum()}")
print(f"Test:  {test_df.shape} (M?a {test_df['season'].min()}-{test_df['season'].max()}), All-Stars: {test_df['All_Star'].sum()}")